# Gaussian Naive Bayes from Scratch

Gaussian Naive Bayes is a probabilistic classification algorithm based on Bayes' Theorem.

It assumes that:

1. Features are conditionally independent given the class label.
2. Continuous features follow a Gaussian (Normal) distribution within each class.

### Bayes' Theorem

P(C|X) ∝ P(C) × P(X|C)

where:

- P(C|X) = Posterior Probability
- P(C) = Prior Probability
- P(X|C) = Likelihood
- P(X) = Evidence (constant for all classes)

### Training Process

For each class:

1. Compute the class prior probability.
2. Compute the mean of each feature.
3. Compute the variance of each feature.

These statistics define the Gaussian distribution for every feature within each class.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Class 0
X0 = np.random.normal(
    loc=[2, 2],
    scale=[1, 1],
    size=(100, 2)
)

# Class 1
X1 = np.random.normal(
    loc=[6, 6],
    scale=[1, 1],
    size=(100, 2)
)

# Class 2
X2 = np.random.normal(
    loc=[2, 6],
    scale=[1, 1],
    size=(100, 2)
)

X = np.vstack([X0, X1, X2])

y = np.array(
    [0]*100 +
    [1]*100 +
    [2]*100
)

df = pd.DataFrame(
    X,
    columns=["feature1", "feature2"]
)

df["target"] = y

print(df.head())

   feature1  feature2  target
0  2.496714  1.861736       0
1  2.647689  3.523030       0
2  1.765847  1.765863       0
3  3.579213  2.767435       0
4  1.530526  2.542560       0


## Gaussian Probability Density Function (PDF)

For continuous features, Gaussian Naive Bayes models each feature using a Normal Distribution.

The likelihood of a feature value is calculated using the Gaussian Probability Density Function.

The probability is higher when the feature value is closer to the class mean and lower when it is farther away.

This function is used to estimate the likelihood of each feature belonging to a particular class.

## Prediction

For a new query point:

1. Compute the prior probability of each class.
2. Compute the Gaussian likelihood for every feature.
3. Combine the likelihoods with the prior probability.
4. Compute the posterior score for each class.
5. Select the class with the highest posterior probability.

To avoid numerical underflow caused by multiplying many small probabilities, logarithms are used during computation.

In [5]:
classes = np.unique(y)

means = {}
variances = {}
priors = {}

for c in classes:
    X_c = X[y == c]

    means[c] = np.mean(X_c, axis=0)
    variances[c] = np.var(X_c, axis=0)
    priors[c] = len(X_c) / len(X)
def gaussian_pdf(x, mean, variance):
    
    numerator = np.exp(
        -((x - mean) ** 2) / (2 * variance)
    )

    denominator = np.sqrt(
        2 * np.pi * variance
    )

    return numerator / denominator
query_point = np.array([3, 4])

posteriors = []

for c in classes:

    prior = np.log(priors[c])

    likelihood = np.sum(
        np.log(
            gaussian_pdf(
                query_point,
                means[c],
                variances[c]
            )
        )
    )

    posterior = prior + likelihood

    posteriors.append(posterior)

prediction = classes[np.argmax(posteriors)]

print("Predicted Class:", prediction)

Predicted Class: 2


## Gaussian Naive Bayes using Scikit-Learn

To validate the correctness of our implementation, we train a Gaussian Naive Bayes model using Scikit-Learn.

Scikit-Learn automatically:

- Computes class priors
- Estimates feature means
- Estimates feature variances
- Computes posterior probabilities
- Generates predictions

The prediction produced by Scikit-Learn is compared with the prediction obtained from the from-scratch implementation.

In [13]:
from sklearn.naive_bayes import GaussianNB

# Create model
gnb = GaussianNB()

# Train on full dataset
gnb.fit(X, y)

# Query point
query_point = np.array([[3, 4]])

# Prediction
prediction = gnb.predict(query_point)

print("Predicted Class:", prediction[0])

Predicted Class: 2


## Conclusion

Gaussian Naive Bayes is a simple yet powerful probabilistic classifier.

Despite its strong assumption of feature independence, it often performs well in practice and serves as an excellent introduction to probabilistic machine learning algorithms.

This implementation demonstrates how class priors, Gaussian distributions, and Bayes' Theorem work together to perform classification.